Setup & Text Cleaning

In [ ]:
import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text) # Remove punctuation
    words = text.split()
    return " ".join([w for w in words if w not in stop_words])

The Data Pipeline

In [ ]:
def load_data(resume_path, jd_path):
    # Load Job Description
    with open(jd_path, 'r') as f:
        jd = clean_text(f.read())
        
    # Load all Resumes
    resumes = []
    filenames = []
    for file in os.listdir(resume_path):
        if file.endswith('.txt'):
            with open(os.path.join(resume_path, file), 'r') as f:
                resumes.append(clean_text(f.read()))
                filenames.append(file)
    return jd, resumes, filenames

jd, resumes, filenames = load_data('../data/job_descriptions/software_engineer_jd.txt', '../data/sample_resumes/')

Vectorization & Cosine Similarity

In [ ]:
# Combine JD and Resumes for vectorization
all_content = [jd] + resumes

vectorizer = TfidfVectorizer()
matrix = vectorizer.fit_transform(all_content)

# Compare JD (index 0) with all Resumes (index 1 onwards)
scores = cosine_similarity(matrix[0:1], matrix[1:])

Results & Ranking

In [ ]:
results = pd.DataFrame({
    'Candidate': filenames,
    'Score': (scores[0] * 100).round(2)
})

results = results.sort_values(by='Score', ascending=False)
print("--- Resume Ranking Results ---")
print(results)